# 05 · Cross-Sectional Evaluation
Run the full pipeline on a basket of large-cap US stocks and compare the LSTM strategy against buy & hold on the held-out test period.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd
from quant_dl.data import download
from quant_dl.pipeline import run_experiment

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'NVDA', 'META', 'JPM', 'XOM']
START, END = '2018-01-01', '2025-01-01'

In [ ]:
rows = []
cost_tables = {}
for t in TICKERS:
    print(f'--- {t} ---', flush=True)
    df = download(t, START, END)
    r = run_experiment(df, seed=42)
    rows.append({
        'ticker': t,
        'strat_total_return': r['strategy']['total_return'],
        'strat_annualized': r['strategy']['annualized_return'],
        'strat_sharpe': r['strategy']['sharpe_ratio'],
        'strat_max_dd': r['strategy']['max_drawdown'],
        'strat_win_rate': r['strategy']['win_rate'],
        'n_trades': r['strategy']['n_trades'],
        'bh_total_return': r['buy_and_hold']['total_return'],
        'bh_sharpe': r['buy_and_hold']['sharpe_ratio'],
        'bh_max_dd': r['buy_and_hold']['max_drawdown'],
    })
    cost_tables[t] = r['cost_sensitivity']
summary = pd.DataFrame(rows).set_index('ticker')
summary

## Summary
Formatted for the README results table.

In [ ]:
pct_cols = ['strat_total_return', 'strat_annualized', 'strat_max_dd', 'strat_win_rate',
            'bh_total_return', 'bh_max_dd']
fmt = summary.copy()
for c in pct_cols:
    fmt[c] = (fmt[c] * 100).round(1).astype(str) + '%'
for c in ['strat_sharpe', 'bh_sharpe']:
    fmt[c] = fmt[c].round(2)
fmt['n_trades'] = fmt['n_trades'].astype(int)
fmt.to_markdown()

In [ ]:
print(fmt.to_markdown())

## Aggregate
Median across tickers is more robust than the mean to outliers.

In [ ]:
summary.median()

## Transaction-cost sensitivity (example: AAPL)
How the strategy's edge decays as fees rise.

In [ ]:
cost_tables['AAPL']